In [ ]:
# 03_conformal.ipynb

import pandas as pd
import joblib
import numpy as np
from sklearn.model_selection import train_test_split
from mapie.classification import MapieClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv("../data/heart.csv")
X = df.drop("target", axis=1)
y = df["target"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Load trained model (if needed)
try:
    base_model = joblib.load("../models/random_forest.pkl")
except FileNotFoundError:
    base_model = RandomForestClassifier(random_state=42)
    base_model.fit(X_train, y_train)

# Apply conformal prediction
mapie = MapieClassifier(base_model, method="score", cv="prefit")
mapie.fit(X_train, y_train)

y_pred, y_ps = mapie.predict(X_test, alpha=0.05)

# Results
print("Conformal Accuracy:", accuracy_score(y_test, y_pred))

# Show some predictions with prediction sets
for i in range(5):
    print(f"True: {y_test.iloc[i]}, Prediction set: {y_ps[i]}")
